# LSTM: corridas de la revisión del 26 de noviembre de 2026

Usa la misma red, la misma configuración y el mismo procedimiento de la prueba de
`6d_lstm_busqueda_y_test.ipynb` (4 épocas, tres semillas promediadas, reentrenamiento el primer día
de cada mes con todo lo anterior). No busca ni ajusta nada. Solo cambia qué información recibe la
red o en qué periodo se mide:

| Corrida | Información | Periodo | Para qué |
|---|---|---|---|
| A | bloques 1 y 2 (precio y calendario) | 2020-2022 | celda "LSTM con información cortada en t" de la tabla de dos por dos |
| B | bloques 1 a 4 | 2020-2022 | la LSTM de la tesis con reentrenamiento mensual en validación |
| C | bloques 1 y 2 | 2023-2025 y 2026 | la misma celda, exploratoria: ese periodo ya se había abierto |
| D | bloques 1 a 4, precio con 3 días de atraso | 2023-2025 y 2026 | robustez: semana h = 4 a 10 |

Los paneles se leen del repositorio público. Cada mes se guarda apenas termina: si la sesión se
cae, basta volver a correr y sigue donde iba.

In [ ]:
import json
import warnings
from math import erf, sqrt
from pathlib import Path

import keras
import numpy as np
import pandas as pd
from keras import layers

warnings.filterwarnings("ignore", message="Mean of empty slice")

# True: corre en pocos minutos con una red mínima y dos meses por corrida, para revisar que no falle
PRUEBA_RAPIDA = False

REPO = "https://raw.githubusercontent.com/andrw4-4/pronostico-precio-bolsa-colombia/main/paneles/"
SALIDA = Path("resultados/modelo_lstm_revision")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SALIDA = Path("/content/drive/MyDrive/tesis/modelo_lstm_revision")
except ImportError:
    pass
if PRUEBA_RAPIDA:
    SALIDA = SALIDA / "prueba_rapida"
(SALIDA / "pronosticos").mkdir(parents=True, exist_ok=True)

# La configuración congelada de resultados/modelo_lstm_bloques/configuracion_final.json
CONFIG = dict(capas=1, unidades=128, densa=64, dropout=0.2, tasa=0.001, lote=32, ventana=28)
EPOCAS, SEMILLAS = 4, [0, 1, 2]
BLOQUE_3 = ["cmg_despacho", "disp_decl_hoy_rel", "cmg_hoy_vs_precio", "frac_termica_prog"]
BLOQUE_4 = ["brecha_escasez", "presion_hidro_horaria", "aportes_media_hist", "volumen_util_pct", "dias_autonomia"]
CONJUNTOS = {"b12": [], "b12_3_4": BLOQUE_3 + BLOQUE_4}
INICIO, VENTANA_MAX, TECHO = "2015-03-01", 56, "2026-10-06"
PERIODOS = {"2020_2022": ("2020-01-01", "2022-12-31"), "2023_2025": ("2023-01-01", "2025-12-31"),
            "2026": ("2026-01-01", "2026-10-06")}
# corrida: (atraso del precio en días, conjunto de variables, periodos)
CORRIDAS = {"A": (2, "b12", ["2020_2022"]), "B": (2, "b12_3_4", ["2020_2022"]),
            "C": (2, "b12", ["2023_2025", "2026"]), "D": (3, "b12_3_4", ["2023_2025", "2026"])}
if PRUEBA_RAPIDA:
    CONFIG.update(unidades=16, densa=16)
    EPOCAS, SEMILLAS = 1, [0]
    PERIODOS = {"2020_2022": ("2022-11-01", "2022-12-31"), "2023_2025": ("2025-11-01", "2025-12-31"),
                "2026": ("2026-09-01", "2026-10-06")}
print("keras", keras.__version__, "con", keras.backend.backend(), "| salida:", SALIDA)

## 1. Datos

El panel con el precio a 2 días es `panel_lstm_bloques.csv.gz`, el mismo de la prueba. El de 3 días
se arma aquí con `panel_lstm_atraso3.csv.gz` (paso 2e con `ATRASO_PRECIO=3`). A ese panel le falta
`frac_termica_prog`, que el paso 2g solo calculó con 2 días: como es un dato del despacho del día
del pronóstico, con 3 días de atraso es el mismo valor corrido un día. La celda comprueba esa
regla con `cmg_despacho`, que está en los dos paneles.

In [ ]:
def leer(nombre, columnas=None):
    local = Path("paneles") / nombre
    return pd.read_csv(local if local.exists() else REPO + nombre, usecols=columnas,
                       parse_dates=["fecha"] if nombre.endswith(".gz") else None)


INFO = leer("panel_lstm_bloques_variables.csv").set_index("variable")
VARIABLES = CONJUNTOS["b12_3_4"]


def panel_de(atraso):
    hs = list(range(atraso + 1, atraso + 8))
    fijas = (["fecha", "hora", "lp", "dia_semana", "es_festivo"]
             + [f"{c}_k{k}" for k in range(1, 8) for c in ("dia_semana", "es_festivo")] + [f"y_h{h}" for h in hs])
    p2 = leer("panel_lstm_bloques.csv.gz", (fijas if atraso == 2 else ["fecha", "hora", "lp"]) + VARIABLES)
    p2 = p2.sort_values(["fecha", "hora"]).reset_index(drop=True)
    if atraso == 2:
        return p2, hs
    p3 = leer("panel_lstm_atraso3.csv.gz", fijas + [v for v in VARIABLES if v != "frac_termica_prog"])
    p3 = p3.sort_values(["fecha", "hora"]).reset_index(drop=True)
    # lo que es del día del pronóstico R = t + atraso: en el panel de 2 días está en la fila t + 1
    corrido = p2[["fecha", "hora", "cmg_despacho", "frac_termica_prog"]].copy()
    corrido["fecha"] = corrido["fecha"] - pd.Timedelta(days=1)
    p3 = p3.merge(corrido, on=["fecha", "hora"], how="left", validate="one_to_one", suffixes=("", "_corrido"))
    ambos = p3["cmg_despacho"].notna() & p3["cmg_despacho_corrido"].notna()
    assert np.allclose(p3.loc[ambos, "cmg_despacho"], p3.loc[ambos, "cmg_despacho_corrido"], atol=1e-4), (
        "correr un día el panel de 2 días no da el de 3")
    p3 = p3.drop(columns="cmg_despacho_corrido")
    # huecos sueltos: el último valor conocido de esa misma hora, como en el paso 2h
    p3[VARIABLES] = p3.groupby("hora")[VARIABLES].ffill()
    return p3, hs


class Mundo:
    # todo lo que depende del atraso del precio: objetivos, calendario, ingenuo semanal y orígenes
    def __init__(self, atraso):
        self.panel, self.HS = panel_de(atraso)
        p = self.panel
        assert (p.groupby("fecha").size() == 24).all()
        self.f = pd.DatetimeIndex(p["fecha"].drop_duplicates())
        self.n = len(self.f)
        dia = p.iloc[::24].reset_index(drop=True)
        cal = lambda ds, fe: np.column_stack([np.eye(7, dtype=np.float32)[ds], fe.astype(np.float32)])
        self.LP = self.por_dia("lp")
        self.Y = np.stack([self.por_dia(f"y_h{h}") for h in self.HS], axis=1)             # [día, 7, 24]
        self.CAL_T = cal(dia["dia_semana"].to_numpy(), dia["es_festivo"].to_numpy())
        self.CAL_K = np.stack([cal(dia[f"dia_semana_k{k}"].to_numpy(), dia[f"es_festivo_k{k}"].to_numpy())
                               for k in range(1, 8)], axis=1)
        self.Y_SEM = np.full_like(self.Y, np.nan)
        for j, h in enumerate(self.HS):
            k = 7 * int(np.ceil(h / 7)) - h
            self.Y_SEM[k:, j] = self.LP[:self.n - k] - self.LP[k:] if k else 0.0
        self.I0 = int(np.searchsorted(self.f, pd.Timestamp(INICIO)))
        self.PRIMERO = self.I0 + VENTANA_MAX - 1
        self.con_precio = ~np.isnan(self.LP).any(axis=1)
        self.completo = ~np.isnan(self.Y).any(axis=(1, 2))

    def por_dia(self, col):
        return self.panel[col].to_numpy(np.float32).reshape(self.n, 24)

    def origenes(self, desde, hasta, completos=True):
        # días t desde los que se pronostica; con completos=False basta que el primer horizonte caiga en el periodo
        assert pd.Timestamp(hasta) <= pd.Timestamp(TECHO), f"no se puede leer más allá de {TECHO}"
        m = (np.arange(self.n) >= self.PRIMERO) & (self.f >= pd.Timestamp(desde)) & self.con_precio
        if completos:
            m &= self.completo & (self.f + pd.Timedelta(days=max(self.HS)) <= pd.Timestamp(hasta))
        else:
            m &= self.f + pd.Timedelta(days=min(self.HS)) <= pd.Timestamp(hasta)
        return np.flatnonzero(m)

    def entradas(self, variables, hasta):
        # vector de cada día, estandarizado solo con los días de INICIO a `hasta` (el último de entrenamiento)
        partes = [self.LP] + [self.por_dia(v) if INFO.loc[v, "tipo"] == "horaria" else self.por_dia(v)[:, :1]
                              for v in variables]
        M = np.hstack(partes)
        ref = M[self.I0:hasta + 1]
        assert not np.isnan(ref).any()
        M = np.clip((M - ref.mean(axis=0)) / (ref.std(axis=0) + 1e-6), -8, 8)
        return np.hstack([M, self.CAL_T]).astype(np.float32)


MUNDOS = {a: Mundo(a) for a in sorted({c[0] for c in CORRIDAS.values()})}
for a, w in MUNDOS.items():
    print(f"precio a {a} días: {w.n} días, del {w.f[0].date()} al {w.f[-1].date()}, horizontes {w.HS[0]} a {w.HS[-1]}")

## 2. Modelo y procedimiento de la prueba

Las mismas funciones del notebook 6d: codificador y decodificador, pérdida MSE, y cada modelo
mensual entrenado con todos los días cuyos precios objetivo ya se conocen, un número fijo de épocas.

In [ ]:
def construir(n_entradas, c, n_hs):
    historia = keras.Input((c["ventana"], n_entradas))
    semana = keras.Input((n_hs, 8))
    x = layers.Dense(c["densa"], activation="relu")(historia)
    x = layers.Dropout(c["dropout"])(x)
    estados = []
    for _ in range(c["capas"]):
        x, h, s = layers.LSTM(c["unidades"], return_sequences=True, return_state=True)(x)
        estados.append([h, s])
    y = semana
    for estado in estados:
        y = layers.LSTM(c["unidades"], return_sequences=True)(y, initial_state=estado)
    y = layers.Dense(24)(layers.Dropout(c["dropout"])(y))
    modelo = keras.Model([historia, semana], y)
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=c["tasa"]), loss="mse")
    return modelo


def ajustar(w, variables, semilla, tr):
    keras.backend.clear_session()
    keras.utils.set_random_seed(semilla)
    M = w.entradas(variables, tr[-1])
    X = lambda idx: [M[idx[:, None] - np.arange(CONFIG["ventana"] - 1, -1, -1)], w.CAL_K[idx]]
    modelo = construir(M.shape[1], CONFIG, len(w.HS))
    modelo.fit(X(tr), w.Y[tr], epochs=EPOCAS, batch_size=CONFIG["lote"], verbose=0)
    return modelo, X


def con_todo(w, nombre, variables, hasta, ev):
    # entrena con todo lo conocido hasta `hasta` y pronostica los días `ev`; se guarda y no se repite
    ruta = SALIDA / "pronosticos" / f"{nombre}.npy"
    if ruta.exists():
        return np.load(ruta)
    tr = w.origenes(INICIO, hasta)
    assert (w.f[tr] + pd.Timedelta(days=max(w.HS))).max() <= pd.Timestamp(hasta), "un objetivo pasa del corte"
    partes = []
    for s in SEMILLAS:
        modelo, X = ajustar(w, variables, s, tr)
        entrada = X(ev)
        assert not np.isnan(entrada[0]).any()
        partes.append(modelo.predict(entrada, verbose=0))
    pred = np.mean(partes, axis=0)
    np.save(ruta, pred)
    print(f"  {nombre}: {len(tr)} días de entrenamiento, {len(ev)} pronosticados", flush=True)
    return pred


def walk_forward(w, nombre, variables, desde, hasta):
    cortes = pd.date_range(desde, hasta, freq="MS")
    idx, pred = [], []
    for k, corte in enumerate(cortes):
        fin = cortes[k + 1] - pd.Timedelta(days=1) if k + 1 < len(cortes) else pd.Timestamp(hasta)
        ev = w.origenes(corte, hasta, completos=False)
        ev = ev[w.f[ev] <= fin]
        if len(ev) == 0:
            continue
        idx.append(ev)
        pred.append(con_todo(w, f"{nombre}_{corte:%Y_%m}", variables, corte - pd.Timedelta(days=1), ev))
    return np.concatenate(idx), np.concatenate(pred)


def a_filas(w, idx, pred, tope):
    n, hs = len(idx), w.HS
    d = pd.DataFrame({"fecha": np.repeat(w.f[idx], len(hs) * 24), "hora": np.tile(np.arange(1, 25), n * len(hs)),
                      "h": np.tile(np.repeat(hs, 24), n), "y_hat": pred.reshape(-1)})
    return d[d["fecha"] + pd.to_timedelta(d["h"], unit="D") <= pd.Timestamp(tope)]


def media_y_ee(d, L):
    n = len(d)
    u = d - d.mean()
    v = np.mean(u * u)
    for l in range(1, min(L, n - 1) + 1):
        v += 2 * (1 - l / (L + 1)) * np.mean(u[l:] * u[:-l])
    return d.mean(), np.sqrt(v / n)


def criterio(w, idx, pred, tope):
    # la semana completa contra el mejor ingenuo, como en el notebook 6d
    y = w.Y[idx].astype(float)
    fuera = (w.f[idx].to_numpy()[:, None] + np.array(w.HS, dtype="timedelta64[D]")) > np.datetime64(tope)
    y[fuera] = np.nan
    e = y - pred
    ingenuos = {"del día": y, "semanal": y - w.Y_SEM[idx]}
    r_ing = {k: np.sqrt(np.nanmean(v ** 2)) for k, v in ingenuos.items()}
    ref = min(r_ing, key=r_ing.get)
    d = np.nanmean((e ** 2 - ingenuos[ref] ** 2).reshape(len(idx), -1), axis=1)
    m, ee = media_y_ee(d[~np.isnan(d)], max(w.HS) - 1 + 7)
    r = np.sqrt(np.nanmean(e ** 2))
    return dict(rmse_log=r, mejor_ingenuo=ref, rmse_ingenuo=r_ing[ref], mejora=100 * (1 - r / r_ing[ref]),
                dm_t=m / ee, dm_p=1 - erf(abs(m / ee) / sqrt(2)))

## 3. Corridas

Los pronósticos de cada corrida quedan en `corrida_<letra>_<periodo>.csv.gz`, con las columnas
`fecha`, `hora`, `h` y `y_hat`, en las mismas filas de los otros modelos.

In [ ]:
resumen = []
for letra, (atraso, conjunto, periodos) in CORRIDAS.items():
    w = MUNDOS[atraso]
    for periodo in periodos:
        desde, hasta = PERIODOS[periodo]
        print(f"corrida {letra}: {conjunto}, precio a {atraso} días, {periodo}", flush=True)
        idx, pred = walk_forward(w, f"rev{atraso}_{conjunto}_{periodo}", CONJUNTOS[conjunto], desde, hasta)
        filas = a_filas(w, idx, pred, hasta)
        filas.to_csv(SALIDA / f"corrida_{letra}_{periodo}.csv.gz", index=False, float_format="%.6f")
        resumen.append(dict(corrida=letra, atraso=atraso, conjunto=conjunto, periodo=periodo, filas=len(filas),
                            **criterio(w, idx, pred, hasta)))
resumen = pd.DataFrame(resumen)
resumen.to_csv(SALIDA / "resumen.csv", index=False)
json.dump(dict(config=CONFIG, epocas=EPOCAS, semillas=SEMILLAS, corridas=CORRIDAS, periodos=PERIODOS,
               keras=keras.__version__, backend=keras.backend.backend(), prueba_rapida=PRUEBA_RAPIDA),
          open(SALIDA / "corrida.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
resumen.round(4)